# Domain Pretraining di Dataset Gingivitis — Persiapan Bobot untuk Pipeline AC

Notebook ini **tidak** memprediksi skor AC. Tujuannya satu: mengadaptasi backbone
(MobileNetV3) dari bobot ImageNet ke domain **foto intraoral** dengan melatihnya
pada dataset gingivitis, lalu menyimpan bobot backbone-nya untuk dipakai sebagai
inisialisasi di notebook 5 (menggantikan bobot ImageNet).

Alurnya sederhana:

1. Turunkan satu label tingkat-gambar dari anotasi kotak: **keparahan gingivitis
   rata-rata** tiap foto (0–4).
2. Latih backbone pretrained ImageNet untuk mengklasifikasikan keparahan itu.
3. Simpan bobot **backbone** (tanpa kepala klasifikasi) ke `models/`.

Catatan jujur: gingivitis menilai gusi, bukan susunan gigi — jadi labelnya *tidak
langsung* relevan dengan IOTN-AC. Manfaatnya adalah mengadaptasi backbone ke warna,
pencahayaan, dan tekstur foto intraoral, yang umumnya inisialisasi lebih baik
daripada ImageNet untuk data AC yang kecil. Apakah benar membantu, diuji di
notebook 5 dengan membandingkan val-MAE terhadap inisialisasi ImageNet.

In [1]:
# ============ Setup ============
import os, glob, json, time, collections
import numpy as np
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image

# --- Path dataset gingivitis (ubah bila folder dipindah) ---
GINGIVITIS_DIR = '/Users/rdrusdiati/Downloads/A DENTAL INTRAORAL IMAGE DATASET OF GINGIVITIS FOR IMAGE CAPTIONING/Dataset'

# --- Akar proyek IOTN-AC (untuk menyimpan bobot ke models/) ---
try:    _HERE = os.path.dirname(os.path.abspath(__file__))
except NameError: _HERE = os.getcwd()
PROJECT_ROOT = os.path.dirname(_HERE) if os.path.basename(_HERE) == 'notebooks' else _HERE
MODELS_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)

PERANGKAT = ('cuda' if torch.cuda.is_available()
             else ('mps' if getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available()
                   else 'cpu'))
BACKBONE = 'mobilenet_v3_small'   # backbone yang dipretraining lalu dipakai di notebook 5
N_KELAS  = 5                       # tingkat keparahan gingivitis 0..4

assert os.path.isdir(GINGIVITIS_DIR), f'Folder dataset tidak ditemukan:\n{GINGIVITIS_DIR}'
print('Perangkat :', PERANGKAT)
print('Backbone  :', BACKBONE)
print('Simpan ke :', os.path.relpath(MODELS_DIR, PROJECT_ROOT) + '/')

Perangkat : mps
Backbone  : mobilenet_v3_small
Simpan ke : models/


## Label tingkat-gambar dari anotasi kotak

Tiap `.txt` berisi kotak per gigi berformat YOLO: `kelas x y w h`. Kelas 0–1 menandai
area rahang atas/bawah; kelas 2–6 menandai keparahan gingivitis tiap gigi
(2 = non-inflamed … 6 = gingivitis 4), sehingga `kelas − 2` memberi severity 0–4.

Sebagai label satu-angka per foto dipakai **rata-rata severity gigi (dibulatkan)** —
lebih seimbang daripada memakai nilai maksimum, sehingga sinyal pelatihannya lebih kuat.

In [2]:
# ============ Turunkan label severity per foto ============
def severity_gambar(txt):
    s = [int(l.split()[0]) - 2 for l in open(txt) if l.split() and int(l.split()[0]) >= 2]
    return int(round(sum(s) / len(s))) if s else 0

def muat_split(split):
    pasangan = []
    for t in sorted(glob.glob(os.path.join(GINGIVITIS_DIR, split, 'Labels', '*.txt'))):
        stem = os.path.splitext(os.path.basename(t))[0]
        img = os.path.join(GINGIVITIS_DIR, split, 'Images', stem + '.jpg')
        if os.path.exists(img):
            pasangan.append((img, severity_gambar(t)))
    return pasangan

DATA = {s: muat_split(sp) for s, sp in [('train','Training'), ('val','Validation'), ('test','Test')]}
for s in DATA:
    dist = collections.Counter(y for _, y in DATA[s])
    print(f'{s:5s}: {len(DATA[s]):4d} foto | sebaran severity {dict(sorted(dist.items()))}')

train:  732 foto | sebaran severity {0: 2, 1: 93, 2: 196, 3: 346, 4: 95}
val  :  182 foto | sebaran severity {1: 16, 2: 90, 3: 58, 4: 18}
test :  182 foto | sebaran severity {1: 15, 2: 99, 3: 58, 4: 10}


In [3]:
# ============ Dataset & DataLoader ============
RATA, SIMPANG = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]   # normalisasi ImageNet
tf_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(), transforms.Normalize(RATA, SIMPANG)])
tf_eval = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(), transforms.Normalize(RATA, SIMPANG)])

class DataGingivitis(Dataset):
    def __init__(self, pasangan, tf): self.pasangan = pasangan; self.tf = tf
    def __len__(self): return len(self.pasangan)
    def __getitem__(self, i):
        p, y = self.pasangan[i]
        return self.tf(Image.open(p).convert('RGB')), y

def buat_loader(split, tf, shuffle):
    return DataLoader(DataGingivitis(DATA[split], tf), batch_size=32,
                      shuffle=shuffle, num_workers=0)   # num_workers=0 agar aman di notebook

dl_train = buat_loader('train', tf_train, True)
dl_val   = buat_loader('val',   tf_eval,  False)
dl_test  = buat_loader('test',  tf_eval,  False)
print('batch latih:', len(dl_train), '| batch val:', len(dl_val), '| batch test:', len(dl_test))

batch latih: 23 | batch val: 6 | batch test: 6


In [4]:
# ============ Model: backbone pretrained + kepala klasifikasi 5 kelas ============
def buat_backbone(nama, n_kelas, dropout=0.2):
    m = getattr(models, nama)(weights='DEFAULT')          # bobot ImageNet
    if nama.startswith('mobilenet'):
        in_feat = m.classifier[0].in_features
    else:                                                  # efficientnet: (Dropout, Linear)
        in_feat = m.classifier[1].in_features
    m.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_feat, n_kelas))
    return m

model = buat_backbone(BACKBONE, N_KELAS).to(PERANGKAT)
print(f'{BACKBONE}: {sum(p.numel() for p in model.parameters()):,} parameter')

mobilenet_v3_small: 929,893 parameter


In [5]:
# ============ Pelatihan ============
# Data timpang -> bobot kelas berbanding terbalik dengan jumlah contoh.
cnt = collections.Counter(y for _, y in DATA['train'])
w = torch.tensor([1.0 / max(cnt.get(k, 0), 1) for k in range(N_KELAS)], dtype=torch.float32)
w = w / w.mean()
kriteria = nn.CrossEntropyLoss(weight=w.to(PERANGKAT))
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)

@torch.no_grad()
def akurasi(dl):
    model.eval(); benar = tot = 0
    for x, y in dl:
        pred = model(x.to(PERANGKAT)).argmax(1).cpu()
        benar += (pred == y).sum().item(); tot += len(y)
    return benar / max(tot, 1)

EPOCH = 8
terbaik, bobot_terbaik = 0.0, None
print(f'Melatih {EPOCH} epoch di {PERANGKAT}...\n')
for ep in range(EPOCH):
    model.train(); t0 = time.time()
    for x, y in dl_train:
        opt.zero_grad(set_to_none=True)
        loss = kriteria(model(x.to(PERANGKAT)), y.to(PERANGKAT))
        loss.backward(); opt.step()
    av = akurasi(dl_val); tanda = ''
    if av > terbaik:
        terbaik = av
        bobot_terbaik = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        tanda = '  <- terbaik'
    print(f'  epoch {ep+1}/{EPOCH}  val_acc {av:.3f}  ({time.time()-t0:.0f}s){tanda}')

model.load_state_dict(bobot_terbaik)
print(f'\nAkurasi val terbaik : {terbaik:.3f}')
print(f'Akurasi test        : {akurasi(dl_test):.3f}')
print('(Angka ini hanya untuk memastikan backbone benar-benar belajar sesuatu;')
print(' tugas sebenarnya tetap prediksi AC di notebook 5.)')

Melatih 8 epoch di mps...

  epoch 1/8  val_acc 0.115  (112s)  <- terbaik
  epoch 2/8  val_acc 0.099  (103s)
  epoch 3/8  val_acc 0.088  (103s)
  epoch 4/8  val_acc 0.088  (106s)
  epoch 5/8  val_acc 0.110  (104s)
  epoch 6/8  val_acc 0.148  (104s)  <- terbaik
  epoch 7/8  val_acc 0.110  (104s)
  epoch 8/8  val_acc 0.126  (103s)

Akurasi val terbaik : 0.148
Akurasi test        : 0.121
(Angka ini hanya untuk memastikan backbone benar-benar belajar sesuatu;
 tugas sebenarnya tetap prediksi AC di notebook 5.)


In [6]:
# ============ Simpan bobot BACKBONE untuk pipeline AC ============
# Hanya bobot backbone (features.*) yang disimpan; kepala klasifikasi gingivitis dibuang.
sd = model.state_dict()
sd_backbone = {k: v for k, v in sd.items() if not k.startswith('classifier')}
jalur = os.path.join(MODELS_DIR, f'pretrained_gingivitis_{BACKBONE}.pt')
torch.save({'backbone': BACKBONE,
            'state_dict': sd_backbone,
            'val_acc': float(terbaik),
            'sumber': 'gingivitis severity 0-4 (rata-rata dibulatkan)',
            'jumlah_kunci': len(sd_backbone)}, jalur)
print('Bobot backbone disimpan ->', os.path.relpath(jalur, PROJECT_ROOT))
print('jumlah tensor backbone   :', len(sd_backbone))

Bobot backbone disimpan -> models/pretrained_gingivitis_mobilenet_v3_small.pt
jumlah tensor backbone   : 240


## Cara memakainya di notebook 5

Setelah membangun model transfer di notebook 5 (`buat_model(nama)`), muat bobot
backbone hasil pretraining ini **sebelum** fine-tuning, menggantikan inisialisasi
ImageNet pada bagian `features`:

```python
ckpt = torch.load('models/pretrained_gingivitis_mobilenet_v3_small.pt')
m = buat_model('mobilenet_v3_small')                       # backbone + custom head
tak_cocok = m.load_state_dict(ckpt['state_dict'], strict=False)
print('kunci tak terisi:', tak_cocok.missing_keys[:3], '...')   # hanya custom head, wajar
```

`strict=False` hanya mengisi bobot yang cocok (semua `features.*`), sementara custom
head dibiarkan apa adanya. Lalu jalankan pelatihan seperti biasa dan bandingkan
val-MAE terhadap versi ImageNet. Bila lebih baik, domain pretraining terbukti membantu;
bila tidak, cukup kembali ke inisialisasi ImageNet.

Untuk memretraining backbone lain (mis. `efficientnet_b0`), ubah `BACKBONE` di cell
Setup lalu jalankan ulang — berkasnya tersimpan dengan nama backbone masing-masing.